# 11. (2 points)

Given the following information as of December 31, 2018:

In [1]:
import pandas as pd

# Values in thousands ($000).
df = pd.DataFrame(data={
    "Accident Year": [2013, 2014, 2015, 2016, 2017, 2018],
    "Incurred Losses": [4400, 5000, 2800, 3000, 4000, 750],
    "Chain Ladder Ultimate": [4400, 6000, 4000, 5400, 10000, 3000],
    "Bornhuetter-Ferguson Ultimate": [4400, 5833, 4300, 5222, 7000, 4500],
    "Bayesian Ultimate": [4400, 5840, 4273, 5301, 7750, None]
})

print(df)

   Accident Year  Incurred Losses  Chain Ladder Ultimate  \
0           2013             4400                   4400   
1           2014             5000                   6000   
2           2015             2800                   4000   
3           2016             3000                   5400   
4           2017             4000                  10000   
5           2018              750                   3000   

   Bornhuetter-Ferguson Ultimate  Bayesian Ultimate  
0                           4400             4400.0  
1                           5833             5840.0  
2                           4300             4273.0  
3                           5222             5301.0  
4                           7000             7750.0  
5                           4500                NaN  


- $\alpha_{2018} = 12500$
- $\beta_{2018} = 2.5$
- $\phi_{2018} = 1.1$

* a. (0.5 point) Compare the variance of the prior distributions for accident years 2016 and 2017.
* b. (0.5 point) Calculate the Bayesian ultimate loss estimate for accident year 2018.
* c. (0.5 point) Calculate the accident year 2014 correltaed chain ladder log mean ultimate loss given the following parameters:
    * $\alpha_{2013} = 8.5172$
    * $\alpha_{2014} = 8.5172$
    * $\beta_6 = 0$
    * $\rho = 0.2$
* d. (0.5 point) Briefly describe one advantage of each of the following methods:
    * Correlated chain ladder method
    * Bayesian method

## Solution (Verrall, *Obtaining Predictive Distributions for Reserves Which Incorporate Expert Opinion*, Sec. 5; Meyers 2015, Sec. 5)

Verrall's Bayesian chain ladder (gamma prior on the row parameter, $x_i\sim\Gamma(\alpha_i,\beta_i)$, prior mean $M_i=\alpha_i/\beta_i$, prior variance $\alpha_i/\beta_i^2=M_i/\beta_i$) gives a credibility mixture of chain ladder and Bornhuetter-Ferguson:

$$\text{Bayesian}=Z\cdot\text{CL}+(1-Z)\cdot\text{BF},\qquad Z=\frac{p}{\beta\phi+p},$$

where $p=\sum y_k$ is the proportion reported (the inverse of the chain ladder CDF). The same $Z$ applies to the reserve and to the ultimate, since both estimates share the reported losses. A larger $\beta$ (smaller prior variance) moves the answer toward BF.

In [2]:
import numpy as np

d = df.set_index("Accident Year").copy()
d["p"] = d["Incurred Losses"] / d["Chain Ladder Ultimate"]                       # % reported = 1 / CDF
d["M"] = (d["Bornhuetter-Ferguson Ultimate"] - d["Incurred Losses"]) / (1 - d["p"])  # implied prior mean (a priori ultimate)
print(d[["p", "M"]].round(4))

# a. back out Z and beta*phi for 2016, 2017 from the given Bayesian ultimates
cl_u, bf_u, by_u = (d[c] for c in ["Chain Ladder Ultimate", "Bornhuetter-Ferguson Ultimate", "Bayesian Ultimate"])
Z = (by_u - bf_u) / (cl_u - bf_u)
bphi = d["p"] * (1 - Z) / Z                  # from Z = p / (beta*phi + p)
print(pd.DataFrame({"Z": Z, "beta*phi": bphi}).loc[[2016, 2017]].round(4))
ratio = bphi[2017] / bphi[2016]              # Var = M/beta ~ 1/(beta*phi) for equal M and phi
print("Var(prior 2016) / Var(prior 2017) =", round(ratio, 3), "(equal phi assumed)")

# b. AY2018
alpha, beta, phi = 12500, 2.5, 1.1
p18 = d.loc[2018, "p"]
Z18 = p18 / (beta * phi + p18)
bayes_2018 = Z18 * cl_u[2018] + (1 - Z18) * bf_u[2018]
print(f"M_2018 = {alpha/beta:.0f} (check: BF = {750 + alpha/beta*(1-p18):.0f}), Z = {Z18:.4f}, Bayesian ultimate = {bayes_2018:.1f}")
assert np.isclose(bayes_2018, 4375.0)

# c. correlated chain ladder log mean, AY2014 ultimate (d = 6)
a2013 = a2014 = 8.5172
beta6, rho = 0.0, 0.2
mu_2013 = a2013 + beta6
mu_2014 = a2014 + beta6 + rho * (np.log(d.loc[2013, "Incurred Losses"]) - mu_2013)
print(f"c. mu_2014,6 = {mu_2014:.4f}  (exp = {np.exp(mu_2014):,.0f})")
assert np.isclose(mu_2014, 8.4916, atol=1e-4)

                    p       M
Accident Year                
2013           1.0000     NaN
2014           0.8333  4998.0
2015           0.7000  5000.0
2016           0.5556  4999.5
2017           0.4000  5000.0
2018           0.2500  5000.0
                    Z  beta*phi
Accident Year                  
2016           0.4438    0.6962
2017           0.2500    1.2000
Var(prior 2016) / Var(prior 2017) = 1.724 (equal phi assumed)
M_2018 = 5000 (check: BF = 4500), Z = 0.0833, Bayesian ultimate = 4375.0
c. mu_2014,6 = 8.4916  (exp = 4,874)


**a.** The implied prior mean (a priori ultimate) is $M=5{,}000$ for both years (from $BF=\text{incurred}+M(1-p)$). Backing $Z$ out of the Bayesian ultimates gives $Z_{2016}=79/178=0.444$ and $Z_{2017}=0.25$. With $Z=p/(\beta\phi+p)$ and $p_{2016}=0.556$, $p_{2017}=0.4$: $\beta\phi=0.696$ (2016) and $1.2$ (2017). Prior variance is $M/\beta$, so, assuming the same $\phi$ for both years, $Var_{2016}/Var_{2017}=\beta_{2017}/\beta_{2016}=1.2/0.696\approx$ **1.72**. The **2016 prior has the larger variance** (less certain prior, so it gets more credibility given to the chain ladder, $Z=0.44$ vs $0.25$). The conclusion does not depend on $\phi$ if it is equal for both years; $\phi$ is only given for 2018.

**b.** $p=750/3000=0.25$; $M=12{,}500/2.5=5{,}000$ (consistent with BF = 750 + 5,000(0.75) = 4,500). $Z=0.25/(2.5\times1.1+0.25)=0.0833$. Bayesian ultimate $=0.0833(3{,}000)+0.9167(4{,}500)=$ **4,375 ($000)**.

**c.** In the correlated chain ladder (Meyers), $\mu_{w,d}=\alpha_w+\beta_d+\rho\,[\ln C_{w-1,d}-\mu_{w-1,d}]$ for $w>1$. AY 2013 is fully developed ($C_{2013,6}=4{,}400$), $\mu_{2013,6}=8.5172$.
$\mu_{2014,6}=8.5172+0+0.2\,(\ln4400-8.5172)=8.5172+0.2(8.3894-8.5172)=$ **≈ 8.4916** (exp = about 4,874).

**d.**
- **Correlated chain ladder:** it allows correlation between accident years (and treats the accident-year level as random), which raises the predicted variability so that the predictive distribution is less understated than in the Mack model (which underestimates variability and validates poorly). It gives a full predictive distribution of outcomes.
- **Bayesian method (Verrall):** it lets the actuary bring in expert opinion through the prior (mean $M_i$ and variance via $\beta_i$) and spans the whole range between chain ladder and Bornhuetter-Ferguson through the credibility factor $Z$. It also gives a full predictive distribution with parameter uncertainty reflected.

## chainladder-python gaps

- No **Verrall Bayesian chain ladder** / credibility factor $Z=p/(\beta\phi+p)$ between chain ladder and BF. `cl.Benktander` and `cl.BornhuetterFerguson` need an a priori and have no prior variance or credibility parameter ($\beta$, $\phi$). I computed $Z$ and the ultimate by hand.
- No **Bayesian/MCMC reserving** (no priors, posterior sampling or predictive distribution). `cl.BootstrapODPSample` and `cl.MackChainladder` are the only stochastic options.
- No **Meyers LCL/CCL** (lognormal incurred models with random accident-year level and between-year correlation $\rho$), and no inversion helper to back out the credibility $Z$ implied by given ultimates.
- Suggested feature: a `BayesianChainladder(prior_mean=..., beta=..., phi=...)` (closed form, with `credibility_`, `ultimate_`, `ibnr_`, `prior_variance_`); the sketch below shows the closed-form piece. Longer term, an MCMC backend for CCL.

## Sketch: `VerrallBayesian`

In [3]:
from sklearn.base import BaseEstimator


class VerrallBayesian(BaseEstimator):
    """Closed-form Verrall Bayesian chain ladder: Z*CL + (1-Z)*BF per origin."""

    def __init__(self, beta, phi):
        self.beta, self.phi = beta, phi              # scalars or per-origin arrays

    def fit(self, latest, cl_ultimate, prior_mean):
        latest, cl_ultimate, prior_mean = (np.asarray(v, dtype=float) for v in (latest, cl_ultimate, prior_mean))
        self.pct_reported_ = latest / cl_ultimate                       # 1 / CDF
        self.credibility_ = self.pct_reported_ / (np.asarray(self.beta) * self.phi + self.pct_reported_)
        self.bf_ultimate_ = latest + prior_mean * (1 - self.pct_reported_)
        self.cl_ultimate_ = cl_ultimate
        self.ultimate_ = self.credibility_ * cl_ultimate + (1 - self.credibility_) * self.bf_ultimate_
        self.ibnr_ = self.ultimate_ - latest
        self.prior_variance_ = prior_mean / np.asarray(self.beta)       # alpha / beta^2
        return self


est = VerrallBayesian(beta=2.5, phi=1.1).fit([750], [3000], [12500 / 2.5])
print(est.credibility_, est.ultimate_)
assert np.isclose(est.ultimate_[0], bayes_2018)

[0.08333333] [4375.]


Extensions not prototyped: origin-specific $\beta_i$, $\phi$ estimated from the data, the posterior variance (Verrall Sec. 5), and the MCMC CCL model (Meyers Sec. 5).